# MCP Concepts

**Prerequisites:** from `03_tools`: `01_what_is_a_tool.ipynb`, `02_the_registry.ipynb`. From `02_agent_runtime`: `02_the_agent_loop.ipynb`

By now an agent can use tools. `03_tools` built `Tool`, which reads a function's signature and produces both the JSON the model sees and the check on the arguments that come back, and `Registry`, which holds a lot of them at once. All of it is Python, all of it in one process, all of it imported.

The Model Context Protocol is usually introduced as a new way to give a model tools. That framing makes it sound like a competitor to what chapter 3 built, and invites the obvious question: why would anyone define a tool over a wire protocol when a Python function already works?

This notebook answers that question by first showing that **the premise is wrong**. MCP is not a different way to describe a tool to a model. The description is the same description. What changes is who owns the process the tool runs in — and every interesting consequence in this chapter follows from that one fact rather than from anything about the protocol's design.

So the order here is: show that the two are identical where you would expect them to differ, find the place where they actually differ, and then let the rest of the chapter be about what that costs.

## Setup

`client.py` sits one folder up and holds the connection details — which interpreter launches the server, from which directory. Those are dull but they are also where a stdio connection goes wrong on Windows, so they live in one place rather than in every notebook. `05_mcp_transport.ipynb` is where they get taken apart.

`03_tools` comes in too, because the whole first half of this notebook is a comparison against it.

In [1]:
import json
import os
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
sys.path.append(str(Path.cwd().parent.parent / "03_tools"))

from client import SERVER_LOG, connect
from tools import Tool

## The tool we already have

Chapter 3's example. `weather` is imported rather than written out here, and that matters more than it looks: the demo server exposes **this same function object**. If it were typed out twice, the comparison two cells from now would prove only that someone had kept two copies in step — which is the exact failure `03_tools` opens with.

A plain function, and a `Tool` wrapped around it:

In [2]:
from demo_tools import weather

local = Tool(weather)
print(json.dumps(local.schema, indent=2))

{
  "type": "function",
  "function": {
    "name": "weather",
    "description": "Look up the current weather in a city.",
    "parameters": {
      "additionalProperties": false,
      "properties": {
        "city": {
          "type": "string"
        },
        "units": {
          "default": "celsius",
          "type": "string"
        }
      },
      "required": [
        "city"
      ],
      "type": "object"
    }
  }
}


That is what goes in the `tools=[...]` list on every request. The model reads it and nothing else — it never sees the function.

## The same tool, in a process we do not own

`server.py`, one folder up, imports that same function and hands it to an MCP server instead of to a `Tool`. Both sides read the description off the docstring too, so nothing in this comparison is being held in step by hand. Now open a session to it.

Note what this is doing: `stdio_params()` describes how to *launch a program*. There is no address and no port. The client starts a second Python process and talks to it down its stdin and stdout.

In [3]:
async with connect() as client:
    listed = await client.list_tools()

print("tools on the server:", [t.name for t in listed.tools])

remote = next(t for t in listed.tools if t.name == "weather")
print()
print("name:       ", remote.name)
print("description:", remote.description)
print("inputSchema:", json.dumps(remote.input_schema, indent=2))

tools on the server: ['weather', 'echo', 'whoami', 'raises', 'described', 'detailed']

name:        weather
description: Look up the current weather in a city.
inputSchema: {
  "type": "object",
  "properties": {
    "city": {
      "title": "City",
      "type": "string"
    },
    "units": {
      "default": "celsius",
      "title": "Units",
      "type": "string"
    }
  },
  "required": [
    "city"
  ],
  "title": "weatherArguments"
}


Every cell in this notebook that talks to the server opens its own session like that, which is
not how you would write an agent and is worth explaining once.

A session cannot span notebook cells. It survives across them well enough to answer calls, but
closing it from a *different* cell raises `RuntimeError: Attempted to exit cancel scope in a
different task than it was entered in`. The session's lifetime is managed by anyio cancel
scopes, those are bound to the task that opened them, and ipykernel runs every cell in a fresh
task — so the block that opened the session and the block that closes it are, as far as anyio
is concerned, two unrelated places.

The cost of doing it per cell is a Python process launched and shut down each time, which is
slow and completely correct. Results that need to outlive the block are kept as plain data —
`listed` above is an ordinary object and does not care that its session is gone.

## These are the same JSON

Hold the two descriptions of `weather` next to each other. The envelopes differ — chapter 3 produces the OpenAI shape, `{"type": "function", "function": {...}}`, while MCP is flat with the schema under `inputSchema` — but that is packaging, and either one converts to the other without losing anything.

What matters is the part the model actually reads to decide how to call the tool:

In [4]:
mine = local.schema["function"]["parameters"]
theirs = remote.input_schema

for label, schema in (("03_tools", mine), ("MCP", theirs)):
    print(f"{label:9}", json.dumps(schema, sort_keys=True))

print()
keys = set(mine) | set(theirs)
for k in sorted(keys):
    a, b = mine.get(k, "--"), theirs.get(k, "--")
    print(f"  {k:20} {'same' if a == b else 'differs'}")

03_tools  {"additionalProperties": false, "properties": {"city": {"type": "string"}, "units": {"default": "celsius", "type": "string"}}, "required": ["city"], "type": "object"}
MCP       {"properties": {"city": {"title": "City", "type": "string"}, "units": {"default": "celsius", "title": "Units", "type": "string"}}, "required": ["city"], "title": "weatherArguments", "type": "object"}

  additionalProperties differs
  properties           differs
  required             same
  title                differs
  type                 same


**The model cannot tell these apart, because there is nothing to tell apart.** Same type, same properties, same required list. A model handed the MCP version and a model handed chapter 3's version are doing exactly the same inference.

The two differences that do show up are both small and both revealing:

`additionalProperties: false` is on chapter 3's version and not on MCP's. That comes from `args_model()` setting `extra="forbid"`, which is what makes an unexpected argument a loud failure rather than a silently ignored one. MCP does not require it, so the SDK does not emit it — which means a foreign server's schema is, by default, more permissive than one we would have written.

`title` is on MCP's and not on ours. Pydantic adds `"title": "City"` to a field already named `city`, and [`Tool.schema`](../../03_tools/tools.py) deliberately strips those, because they say nothing and are resent on every turn of every conversation. The MCP server generates its schema straight from Pydantic and keeps them.

Neither changes what the model does. Both are worth noticing now because they are the first hint of the real theme: **on the far side of the boundary, decisions you made carefully are made by somebody else.**

## So what is actually different

Not the JSON. This:

In [5]:
print("this notebook: pid", os.getpid())

async with connect() as client:
    r = await client.call_tool("whoami", {})

print("the server:   ", r.content[0].text)

this notebook: pid 15472


the server:    pid=19932 executable=C:\Users\91879\anaconda3\envs\agentic-forge\python.exe


Two processes. That is the whole of it, and everything else in this chapter is a consequence.

The server is a separate program. It was not imported — it was *launched*. It has its own interpreter, its own memory, its own imports, its own crashes. It could have been written in TypeScript. It could be running on another machine. Nothing in this notebook would change.

Compare what each arrangement actually gives you:

| | A Python function as a tool (`03_tools`) | The same tool over MCP |
|---|---|---|
| How the agent gets it | `import` | launch a process, or open a connection |
| What the model sees | the schema above | the same schema |
| Who can write it | someone writing Python, in your repo | anyone, in any language, in their repo |
| When you find out its tools changed | at import, at your next deploy | at runtime, on this connection |
| What a crash in it takes down | your agent | itself |
| What you can inspect when it fails | the whole traceback | whatever it chose to tell you |

The first four rows are why MCP exists. The last two are what it costs, and the rest of this notebook is about the last two.

## Three roles, not two

The diagram people usually draw has an agent and a server. There is a third role in between, and it matters because it is the one this notebook is playing:

```
Host                        ← decides which servers exist, and what the model may see
  │
  ├── Client ──────┐        ← one per server, owns one session
  │                │ MCP protocol (JSON-RPC 2.0)
  │                ▼
  │            Server       ← a separate process
  │              ├── Tools      (things the model can call)
  │              ├── Resources  (data the host can read)
  │              └── Prompts    (templates a user can pick)
  │
  └── Client ──→ another Server
```

The **server** exposes capabilities. The **client** holds exactly one session with exactly one server — two servers means two clients. The **host** owns the clients and makes every decision the protocol deliberately does not: which servers to connect to at all, which of their tools to put in front of the model, what to do when one of them fails.

That last role is the one with no code in the SDK, and it is where your agent lives. When `06_build_an_mcp_agent.ipynb` wires this into `02_agent_runtime`'s loop, the runtime is the host — and `Registry` is the thing that decides which discovered tools the model is allowed to see.

This is also the honest answer to "is MCP a framework?" It is not. It specifies the wire between client and server and says nothing about how you build an agent, which is why chapter 2's loop needs no modification to use it.

## Sessions

The connection is not stateless. Opening it ran a handshake — that is what `enter_async_context` did several cells ago, and it is why `list_tools()` worked without any setup.

Three things were settled in it:

In [6]:
async with connect() as client:
    print("protocol version:", client.protocol_version)
    print("server:          ", client.server_info.name, client.server_info.version)
    print()
    print("capabilities the server declared:")
    for field, value in client.server_capabilities:
        if value is not None:
            print(f"  {field:14} {value}")

protocol version: 2026-07-28
server:           forge-demo 0.1.0

capabilities the server declared:
  prompts        list_changed=True
  resources      subscribe=True list_changed=True
  tools          list_changed=True


**Protocol version** is negotiated, not assumed. Client and server each state what they speak and settle on a common version, which is what lets a client from last year talk to a server from this one.

**Server identity** is a name and a version string. Worth noting for what it is *not*: there is nothing here that authenticates anything. A server calling itself `forge-demo` is a server that says it is `forge-demo`.

**Capabilities** say which parts of the protocol this server implements at all. If a server does not declare `resources`, asking it for resources is a protocol error rather than an empty answer.

None of this is free, and the price is paid per session. Every `async with connect()` in this notebook launched a process and ran that handshake before it could ask a single question — which is fine here and is the reason a real agent opens a session once and keeps it, rather than per tool call.

## Capability, and contents

The server above declared `tools`, `resources` and `prompts`. That reads like it has all three. Ask it:

In [7]:
async with connect() as client:
    resources = await client.list_resources()
    prompts = await client.list_prompts()

print("resources:", resources.resources)
print("prompts:  ", prompts.prompts)

resources: []
prompts:   []


Both empty, and both were declared.

**A capability is a statement about the protocol, not an inventory.** The server is saying "I implement `resources/list`", and the honest answer it gives to that call is an empty list. A host that reads capabilities as a description of contents will build a menu of three things and find two of them empty.

This gap is worth meeting here because it is not a quirk of the demo server. In practice most MCP servers in the wild are tools-only. Resources and prompts are specified, implemented by the SDKs, and thinly used:

- a **tool** is something the *model* decides to call
- a **resource** is data the *host* fetches and decides to put in the context window
- a **prompt** is a template a *user* picks, usually from a menu in the host's UI

The reason tools dominate is that only the first of those has an obvious home in an agent loop. The other two need a host that has somewhere to put them — and for a resource, that somewhere is the context window, which is `05_context`'s problem and not the protocol's. `04_tools_resources_prompts.ipynb` takes each one seriously; the thing to carry out of here is that discovering a capability is not discovering a feature.

## Discovery is a runtime fact

The tools listed at the start of this notebook were not known when the notebook was written. There is no import naming them, no schema checked into this repo, nothing that would have failed if the server had offered a different set.

That is the actual break from `03_tools`. There, the set of tools is fixed when the process starts, because it is whatever got imported. Here it is a question asked over a wire, and the answer can differ between one run and the next — the server can change its mind and say so, which is what `list_changed` in its capabilities means.

It cuts both ways, and the second way is easy to miss. An agent whose tools arrive at runtime is an agent whose tools you did not review. `Registry` refusing a name that is already taken stops being bookkeeping the moment the names are coming from someone else's process.

## Invocation

One call, and the shape of what comes back:

In [8]:
async with connect() as client:
    result = await client.call_tool("weather", {"city": "Oslo"})

print("is_error: ", result.is_error)
print("content:  ", result.content)
print("text:     ", result.content[0].text)

is_error:  False
content:   [TextContent(type='text', text='4 degrees celsius, clear skies in Oslo.', annotations=None, meta=None)]
text:      4 degrees celsius, clear skies in Oslo.


Two things in that shape are worth naming.

`content` is a **list**, not a string, and its entries are typed — `TextContent` here, but `ImageContent`, `AudioContent`, `ResourceLink` and `EmbeddedResource` are all legal. A tool can return an image. Chapter 3's `invoke()` ends every call with `str(result)`, which has no way to express that.

`is_error` is a **bool**. That is the entire error vocabulary, and it is the subject of the rest of this notebook.

## What the boundary costs

`03_tools` spent a notebook arguing that "it failed" is not one fact. [`invoke()`](../../03_tools/invoke.py) sorts failures by who can act on them — `bad_arguments`, `refused`, `unverified`, `timeout`, `broken` — so that the loop can ask two questions: should the model see this, and is it safe to call again. A bug in the tool is ours to fix and the model is not told about it. A tool that ran and said no is something the model can work with.

Across a process boundary there is one bool. Here are three tools on the demo server failing in three different ways:

In [9]:
async with connect() as client:
    for name, args in [("raises", {}), ("described", {"city": "Vestby"}), ("detailed", {})]:
        r = await client.call_tool(name, args)
        ours = {k: v for k, v in (r.meta or {}).items() if k.startswith("forge/")}
        print(f"{name:10} is_error={r.is_error}  {r.content[0].text!r}")
        if ours:
            print(f"{'':10} _meta    {ours}")

raises     is_error=True  'Error executing tool raises'
described  is_error=True  'No weather station for Vestby. Try a larger nearby city.'
detailed   is_error=True  'Weather service unavailable.'
           _meta    {'forge/cause': 'ConnectionResetError at pool.py:412, retry 3/3 exhausted', 'forge/kind': 'broken', 'forge/repeatable': False}


Same bool, three very different amounts of information.

`raises` let a `ValueError` escape. The SDK caught it, returned `is_error=True`, and replaced the message with `Error executing tool raises`. The actual cause — `connection refused by upstream weather API` — went to the server's stderr, in a process this notebook does not read. **It is not that the cause was not transmitted. It is that nothing asked the tool author to transmit it.**

`described` caught its own failure and wrote a message for the model to act on, which is `03_tools`' `ToolError` by hand and without the name.

`detailed` used two channels at once. `content` carries `Weather service unavailable.` into the conversation; `_meta` carries the operational detail — `ConnectionResetError at pool.py:412, retry 3/3 exhausted` — alongside it, where the host can read it and the model cannot. That is the closest thing MCP has to `for_the_model`, and note where the decision was made: on the server, by its author.

`_meta` is a free-form dict that rides on any result and is passed through untouched, which is why the keys are namespaced — the SDK puts its own `io.modelcontextprotocol/serverInfo` in that same dict. It is shared ground, not ours.

A note on why it is `_meta` and not the obvious thing. MCP does have a logging channel, where a server sends the host `notifications/message` at a level, and it is the textbook answer to this problem. It is also **deprecated** as of protocol version `2026-07-28` (SEP-2577) — the version this server and client negotiated several cells ago — along with the `roots` capability. So the textbook answer is on the way out, and every `ctx.error()` call raises `MCPDeprecationWarning`. `_meta` is what is left, and it is where notebook 2 will put `Result.kind` anyway, so the two problems have one answer.

So the answer to "how does a client recover the cause" is that **it does not, and there is no protocol mechanism that would let it.** What there is instead is a convention — catch your own errors, put something the model can act on in `content`, put the rest in `_meta` — and the quality of what you get is a property of the server author's discipline rather than of MCP.

Which means a host cannot rely on it. When `02_mcp_client.ipynb` maps a `CallToolResult` back onto chapter 3's `Result`, `is_error=True` has to be given one `kind` chosen without evidence, and the choice is not obvious: guess `broken` and the model never sees `Try a larger nearby city`, so the agent stalls holding advice it was never shown; guess `refused` and a genuine bug costs a few wasted retries. The cheap mistake wins, but it is a guess either way.

## One last thing the boundary owns: the process itself

Every `async with connect()` in this notebook launched a Python process, and every one of them exited when the block did. That is not tidiness on the SDK's part — **a stdio server exits when its stdin closes, and that is the only signal it gets.** There is no shutdown message in the protocol and no port to close.

So a session that is never closed is a process that never stops. The demo server is a few kilobytes and does nothing; a real one might hold a database connection or a browser.

The count is easy to check, because a stdio server has no idea how many siblings it has:

In [10]:
print("server.log exists:", SERVER_LOG.exists())

async with connect() as client:
    before = await client.call_tool("whoami", {})
async with connect() as client:
    after = await client.call_tool("whoami", {})

print("first session: ", before.content[0].text)
print("second session:", after.content[0].text)

server.log exists: True


first session:  pid=14176 executable=C:\Users\91879\anaconda3\envs\agentic-forge\python.exe
second session: pid=25796 executable=C:\Users\91879\anaconda3\envs\agentic-forge\python.exe


Two sessions, two pids, no relationship between them. Each got a fresh process with no memory of the other — so anything a server accumulates during a session is gone when it closes, and any state you need across sessions has to live somewhere the server can reach, which is `04_memory`'s problem arriving in a new place.

That `server.log` is the other half of the arrangement. In a notebook the server's stderr cannot go to the console — see the `JUPYTER` note in [`client.py`](../client.py) for why — so it goes to a file instead. It is worth opening after anything in this chapter behaves strangely, because it is where a stdio server's crashes actually land.

## Where this leaves the chapter

The thing to carry forward is the sentence this notebook spent its length earning: **MCP does not change what a tool is, it changes who owns the process it runs in.**

Everything the rest of the chapter does is downstream of that:

| | |
|---|---|
| `02_mcp_client` | discovered tools become `Tool` objects in a `Registry`, and `Act` runs them unchanged — plus the lossy part, mapping one bool back onto `Result` |
| `03_tools_resources_prompts` | the two primitives this notebook found empty, taken seriously |
| `04_mcp_server` | the other direction: `Registry` exposed over MCP, and what we can fix on our own side |
| `05_mcp_transport` | stdio versus HTTP: process lifecycle, the orphan above, and where the trust boundary actually sits |
| `06_build_an_mcp_agent` | all of it inside chapter 2's loop, which needs no modification |

One prediction worth writing down now, to check in notebook 4. Both gaps this notebook found in the MCP schema — the missing `additionalProperties: false`, the `title` keys nobody needs — are fixable on a server we write ourselves. Neither is fixable on a server we do not. That asymmetry is the chapter's argument in miniature, and it is why `04_mcp_server` comes after `02_mcp_client` rather than before: it is much easier to see what a good server owes its clients once you have had to consume a mediocre one.